In [16]:
import torch
import pickle
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np
import pandas as pd
from numpy import savetxt, loadtxt
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.nn import Linear, BatchNorm1d
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Input, Concatenate, Reshape, Conv1D, Flatten, Dense, Dropout, MultiHeadAttention, LayerNormalization, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from transformers import AutoTokenizer, AutoModel

from tqdm import tqdm
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dropout, Dense, concatenate, BatchNormalization



In [17]:
print(torch.__version__)

2.5.1+cu121


In [18]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [19]:
def split_into_chunks(tokens, chunk_size=512, num_chunks=10):

    chunks = []

    for i in range(num_chunks):

        start = i * chunk_size
        end = start + chunk_size

        chunk = tokens[start:end]

        if len(chunk) < chunk_size:
            chunk = chunk + [0] * (chunk_size - len(chunk))  # pad

        chunks.append(chunk)

    return chunks
def embed_contract(code, tokenizer, model, device):

    tokens = tokenizer(
        code,
        add_special_tokens=False
    )["input_ids"]

    chunks = split_into_chunks(tokens)

    inputs = tokenizer.pad(
        {"input_ids": chunks},
        padding=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        
    mean_pool = outputs.last_hidden_state.mean(dim=1)
    # max_pool  = outputs.last_hidden_state.max(dim=1).values
    # min_pool  = outputs.last_hidden_state.min(dim=1).values
    # sum_pool  = outputs.last_hidden_state.sum(dim=1)
    # chunk_embeddings = torch.cat(
    # [mean_pool, max_pool, min_pool, sum_pool],
    # dim=1
    # )
    contract_embedding = mean_pool.flatten()

    return contract_embedding.cpu().numpy()

def get_codebert_embeddings(texts, tokenizer, model, device='cuda'):

    embeddings = []

    for code in tqdm(texts, desc="Embedding contracts"):

        emb = embed_contract(code, tokenizer, model, device)

        embeddings.append(emb)

    return np.vstack(embeddings)

In [20]:
def load_dataset(label_count):

    data = pd.read_csv("DATASET/multilabel_BILSTM_BERT.csv", index_col=False)
    data = data.drop(columns='Unnamed: 0')
    print(len(data))

    label_columns = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

    if label_count > len(label_columns):
        raise ValueError("Label count exceeds the available labels")

    selected_labels = label_columns[:label_count]

    for column in label_columns:
        data[column] = data[column].apply(lambda x: 1 if x != 0 else 0)

    # Giữ đúng 39645 rows
    #data = data.iloc[:39645]
    #data = data.iloc[:20000]
    #data = data.iloc[:100]
    #data = data.iloc[:39645]
    total_rows = len(data)
    split_point = int(0.8 * total_rows)

    print(split_point)
    print(total_rows - split_point)

    df_train = data.iloc[:split_point]
    df_test = data.iloc[split_point:]
    

    if True:
        X_train_source = np.load("DATASET/X_train_source_minmaxsummean_10chunks_512.npy")
        X_test_source = np.load("DATASET/X_test_source_minmaxsummean_10chunks_512.npy")
        X_train_source = X_train_source.reshape(-1, 10, 4, 768)
        X_test_source = X_test_source.reshape(-1, 10, 4, 768)
        X_train_source = X_train_source[:, :, 0, :]
        X_test_source = X_test_source[:, :, 0, :]
        X_train_source = X_train_source.reshape(-1, 10 * 768)
        X_test_source = X_test_source.reshape(-1, 10 * 768)

    else:

        # =========================
        # CODEBERT EMBEDDING
        # =========================

        source_train = df_train['sourcecode'].tolist()
        source_test = df_test['sourcecode'].tolist()
        

        tokenizer = AutoTokenizer.from_pretrained("./codebert")
        model = AutoModel.from_pretrained("./codebert")

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)


        model.eval()

        print("Generating CodeBERT embeddings...")

        X_train_source = get_codebert_embeddings(source_train, tokenizer, model, device)
        X_test_source = get_codebert_embeddings(source_test, tokenizer, model, device)
        
            
        # texts = df_train['sourcecode'].astype(str).tolist()

        # lengths = [len(tokenizer.tokenize(x)) for x in texts]

        # print("Average tokens:", np.mean(lengths))
        # print("Max tokens:", np.max(lengths))


        print("Load Features CodeBERT success!!")
        np.save("DATASET/X_train_source.npy", X_train_source)
        np.save("DATASET/X_test_source.npy", X_test_source)

    # =========================
    # OPCODE FEATURES (giữ nguyên)
    # =========================

    X_train_opcode = np.array(df_train.iloc[:, 11:].values)
    X_test_opcode = np.array(df_test.iloc[:, 11:].values)

    # =========================
    # LABELS
    # =========================

    y_train = df_train[selected_labels].values
    y_test = df_test[selected_labels].values
    labels = data[selected_labels].values

    print(f"Load Data for {label_count}-Label MultiLabel success!!")

    return X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels

In [21]:
X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels = load_dataset(8)

45597
36477
9120
Load Data for 8-Label MultiLabel success!!


In [22]:
codebert_input = Input(shape=(7680,), name="sourcecode_features")

x = BatchNormalization()(codebert_input)

x = Dense(1024, activation="relu")(x)
x = Dropout(0.3)(x)

x = Dense(512, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)

x = Dense(128, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Dense(64, activation="relu")(x)

codebert_output_final = Dense(8, activation="sigmoid")(x)

In [23]:
# codebert_input = Input(shape=(768,), dtype="float32", name='sourcecode_features')
# # Code trích xuất (vẫn 4 tiếng)

# codebert_dense_1 = Dense(512, name="codebert_dense_layer_1", activation="relu")(codebert_input)
# codebert_bn_1 = BatchNormalization(name="codebert_bn_1")(codebert_dense_1)
# codebert_dropout_1 = Dropout(0.3, name="codebert_dropout_layer_1")(codebert_bn_1)

# codebert_dense_2 = Dense(256, name="codebert_dense_layer_2", activation="relu")(codebert_dropout_1)
# codebert_bn_2 = BatchNormalization(name="codebert_bn_2")(codebert_dense_2)
# codebert_dropout_2 = Dropout(0.3, name="codebert_dropout_layer_2")(codebert_bn_2)

# codebert_dense_3 = Dense(128, name="codebert_dense_layer_3", activation="relu")(codebert_dropout_2)
# codebert_bn_3 = BatchNormalization(name="codebert_bn_3")(codebert_dense_3)
# codebert_dropout_3 = Dropout(0.3, name="codebert_dropout_layer_3")(codebert_bn_3)

# codebert_output_1 = Dense(64, name="codebert_output_1", activation="relu")(codebert_dropout_3)
# codebert_output_final =  Dense(8, name="codebert_output", activation="sigmoid")(codebert_output_1)

In [24]:
model = tf.keras.Model(inputs=codebert_input, outputs=codebert_output_final)
METRICS = [
      tf.keras.metrics.BinaryAccuracy(name='accuracy'),
      tf.keras.metrics.Precision(name='precision'),
      tf.keras.metrics.Recall(name='recall'),
      # tf.keras.metrics.F1Score(name='f1')
]

In [25]:
from tensorflow.keras.losses import CategoricalCrossentropy,  BinaryCrossentropy
import numpy as np

adam = Adam()
model.compile(optimizer=adam, loss=BinaryCrossentropy(),  metrics=METRICS)

model.fit(np.array(X_train_source), y_train, epochs=100, batch_size=32)
# print(len(text))
# print(len(labels))

Epoch 1/100
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 59s 50ms/step - accuracy: 0.7978 - loss: 0.4533 - precision: 0.7374 - recall: 0.5967
Epoch 2/100
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 56s 49ms/step - accuracy: 0.8193 - loss: 0.4079 - precision: 0.7727 - recall: 0.6370
Epoch 3/100
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 54s 47ms/step - accuracy: 0.8274 - loss: 0.3836 - precision: 0.7832 - recall: 0.6558
Epoch 4/100
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 54s 47ms/step - accuracy: 0.8365 - loss: 0.3659 - precision: 0.7929 - recall: 0.6794
Epoch 5/100
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 53s 46ms/step - accuracy: 0.8425 - loss: 0.3526 - precision: 0.8002 - recall: 0.6932
Epoch 6/100
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 58s 50ms/step - accuracy: 0.8491 - loss: 0.3394 - precision: 0.8075 - recall: 0.7093
Epoch 7/100
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 57s 50ms/step - accuracy: 0.8548 - loss: 0.3282 - precision: 0.8154 - recall: 0.7211
Epoch 8/100
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 57s 50ms/step - accuracy: 0.8602 - loss: 0.3171 - precision: 

In [26]:
model.save("DATASET/bert.h5")

In [27]:
#model.fit(np.array(X_train_source), y_train, epochs=50, batch_size=32)

In [28]:
y_pred = model.predict(np.array(X_test_source))

y_test = (y_test >= 1).astype(int)


y_pred = (y_pred > 0.5).astype(int)

285/285 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [29]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
label_names = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

for i, label in enumerate(label_names):
    precision = precision_score(y_test[:, i], y_pred[:, i])
    recall = recall_score(y_test[:, i], y_pred[:, i])
    f1 = f1_score(y_test[:, i], y_pred[:, i])
    accuracy = accuracy_score(y_test[:, i], y_pred[:, i])
    # Tính confusion matrix cho nhãn i
    tn, fp, fn, tp = confusion_matrix(y_test[:, i], y_pred[:, i]).ravel()

    # Tính FPR và FNR32
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    print(f"Metrics for {label}:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  FPR:       {fpr:.4f}")
    print(f"  FNR:       {fnr:.4f}")
    print()

Metrics for Other:
  Accuracy:  0.9172
  Precision: 0.9472
  Recall:    0.9278
  F1-Score:  0.9374
  FPR:       0.1040
  FNR:       0.0722

Metrics for access_control:
  Accuracy:  0.9670
  Precision: 0.8159
  Recall:    0.4748
  F1-Score:  0.6003
  FPR:       0.0059
  FNR:       0.5252

Metrics for arithmetic:
  Accuracy:  0.9424
  Precision: 0.9520
  Recall:    0.9571
  F1-Score:  0.9545
  FPR:       0.0826
  FNR:       0.0429

Metrics for denial_service:
  Accuracy:  0.9712
  Precision: 0.8985
  Recall:    0.8935
  F1-Score:  0.8960
  FPR:       0.0163
  FNR:       0.1065

Metrics for front_running:
  Accuracy:  0.9485
  Precision: 0.8079
  Recall:    0.7063
  F1-Score:  0.7537
  FPR:       0.0211
  FNR:       0.2937

Metrics for reentrancy:
  Accuracy:  0.9169
  Precision: 0.9333
  Recall:    0.8620
  F1-Score:  0.8962
  FPR:       0.0440
  FNR:       0.1380

Metrics for time_manipulation:
  Accuracy:  0.9806
  Precision: 0.8172
  Recall:    0.7904
  F1-Score:  0.8036
  FPR:       

In [30]:
from sklearn.metrics import classification_report

print("CodeBERT")
print(classification_report(y_test, y_pred))

CodeBERT
              precision    recall  f1-score   support

           0       0.95      0.93      0.94      6091
           1       0.82      0.47      0.60       476
           2       0.95      0.96      0.95      5755
           3       0.90      0.89      0.90      1268
           4       0.81      0.71      0.75      1018
           5       0.93      0.86      0.90      3798
           6       0.82      0.79      0.80       458
           7       0.96      0.94      0.95      4039

   micro avg       0.94      0.90      0.92     22903
   macro avg       0.89      0.82      0.85     22903
weighted avg       0.93      0.90      0.92     22903
 samples avg       0.87      0.85      0.85     22903



c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
